# AnimalCLEF 2026: Global + Local Features Ensemble

**Approach**: Combines global features (MiewID v3) with species-specific local features (SIFT, SuperPoint, ALIKED, DISK) using weighted ensemble voting.

**Expected Improvements**:
- SeaTurtleID2022: 38% → 50-60% known match rate
- Better clustering for deformable bodies (Salamanders) and dense patterns (Texas Lizards)

**Runtime**: ~90 minutes (within Kaggle's 9-hour limit)

## Section 1: Setup and Installation

In [ ]:
# Cell 1.1: Install Dependencies
!pip install kornia kornia-rs kornia-moons --quiet
!pip install wildlife-datasets wildlife-tools timm scikit-learn --quiet --upgrade
# Use transformers 4.36.0 for MiewID v3 compatibility
!pip install transformers==4.36.0 --quiet

import kornia
print(f"✓ Kornia {kornia.__version__} installed")

import transformers
print(f"✓ Transformers {transformers.__version__} installed")
print("✓ All dependencies installed successfully")

In [ ]:
# Cell 1.2: Imports
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import normalize
import cv2
from PIL import Image

# Deep learning
import timm
from transformers import AutoModel

# Local features (kornia for all extractors)
import kornia.feature as KF

# Dataset
from wildlife_datasets.datasets import AnimalCLEF2026

warnings.filterwarnings("ignore")
print("✓ Imports successful")

In [ ]:
# Cell 1.3: Configuration - Species-Specific Weights

# SIMPLIFIED: Use only SIFT (proven to work) for local features
# This ensures the notebook runs without kornia compatibility issues

SPECIES_CONFIG = {
    "SalamanderID2025": {
        # Deformable bodies → SIFT only (rotation invariant)
        "global_weight": 0.70,
        "local_extractors": ["sift"],
        "local_weights": {"sift": 0.30},
        "threshold_known": 0.40,
        "threshold_cluster": 0.35,
        "image_size": 512,
        "qe_k": 3,
    },
    "SeaTurtleID2022": {
        # Rigid, high-contrast features → SIFT only
        "global_weight": 0.65,
        "local_extractors": ["sift"],
        "local_weights": {"sift": 0.35},
        "threshold_known": 0.45,
        "threshold_cluster": 0.40,
        "image_size": 512,
        "qe_k": 8,
    },
    "LynxID2025": {
        # Rotation invariance → SIFT (perfect for this)
        "global_weight": 0.70,
        "local_extractors": ["sift"],
        "local_weights": {"sift": 0.30},
        "threshold_known": 0.40,
        "threshold_cluster": 0.35,
        "image_size": 512,
        "qe_k": 5,
    },
    "TexasHornedLizards": {
        # Dense spot patterns → SIFT
        "global_weight": 0.65,
        "local_extractors": ["sift"],
        "local_weights": {"sift": 0.35},
        "threshold_known": None,  # Zero-shot
        "threshold_cluster": 0.30,
        "image_size": 512,
        "qe_k": 5,
    },
}

# Verify weights sum to 1.0
for species, cfg in SPECIES_CONFIG.items():
    total = cfg["global_weight"] + sum(cfg["local_weights"].values())
    assert abs(total - 1.0) < 0.01, f"{species} weights don't sum to 1.0: {total}"

print("✓ Species configuration loaded (SIFT-only for compatibility)")
for species, cfg in SPECIES_CONFIG.items():
    print(f"  {species}: Global {cfg['global_weight']:.0%}, SIFT {cfg['local_weights']['sift']:.0%}")

In [ ]:
# Cell 1.4: Device Detection

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
NUM_WORKERS = 4
ROOT_DIR = "/kaggle/input/animal-clef-2026"

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    if torch.cuda.device_count() > 1:
        print(f"🚀 Using {torch.cuda.device_count()} GPUs") 

# Create cache directory
os.makedirs("cache/embeddings", exist_ok=True)
os.makedirs("cache/local_features", exist_ok=True)
os.makedirs("cache/match_scores", exist_ok=True)
print("✓ Cache directories created")

In [ ]:
# Cell 1.5: Load Data

print("📂 Loading AnimalCLEF 2026 dataset...")
full_dataset = AnimalCLEF2026(root=ROOT_DIR, transform=None, load_label=True)
metadata = full_dataset.metadata

test_meta = metadata[metadata["split"] == "test"]
train_meta = metadata[metadata["split"] == "train"]

print(f"✓ Dataset loaded")
print(f"  Total images: {len(metadata):,}")
print(f"  Test images: {len(test_meta):,}")
print(f"  Train images: {len(train_meta):,}")
print(f"\nSpecies breakdown:")
for species in test_meta["dataset"].unique():
    n_test = len(test_meta[test_meta["dataset"] == species])
    n_train = len(train_meta[train_meta["dataset"] == species])
    print(f"  {species}: {n_test} test, {n_train} train")

In [ ]:
# Cell 1.6: SAM 3 — build SEG_MAP and keypoint-mask helper
#
# Strategy (V1.3): run SIFT on the ORIGINAL image, then discard keypoints
# that land on background pixels.  This avoids the white-boundary artifacts
# that hurt V1.2 (score 0.26330 vs V1 baseline 0.30655).
#
# LynxID2025 has 0% cache coverage → get_seg_mask() returns None → no change
# for that species.  All other species get filtered keypoints.
from pathlib import Path
import cv2
import numpy as np

SEG_CACHE_ROOT = Path('/kaggle/input/datasets/sreevaatsavbavana/animalclef-26-sam3/sam3_yolo_output/segmented_images')
_root = Path(ROOT_DIR)  # ROOT_DIR defined in Cell 1.4 as a str

# Key: relative path  e.g. 'SeaTurtleID2022/img001.jpg'
SEG_MAP = {}  # {rel_key: Path_to_segmented_jpg}

all_meta = pd.concat([train_meta, test_meta])
for _, row in tqdm(all_meta.iterrows(), total=len(all_meta), desc='Building SEG_MAP'):
    stem    = Path(row['path']).stem
    ds_name = row['dataset']
    rel_key = str(row['path'])
    cached  = SEG_CACHE_ROOT / ds_name / (stem + '.jpg')
    if cached.exists():
        SEG_MAP[rel_key] = cached

n_total    = len(all_meta)
n_cached   = len(SEG_MAP)
n_fallback = n_total - n_cached
print(f'SEG_MAP built: {n_cached:,} cached  |  {n_fallback:,} fallback to original  |  {n_total:,} total')

# Per-species / per-split breakdown
print()
print(f'{"Dataset":<25} {"split":<6} {"cached":>7} {"total":>7} {"coverage":>9}')
print('-' * 58)
for ds in sorted(all_meta["dataset"].unique()):
    for split, meta_split in [("train", train_meta), ("test", test_meta)]:
        rows = meta_split[meta_split["dataset"] == ds]
        if len(rows) == 0:
            continue
        n_hit = sum(1 for _, r in rows.iterrows() if str(r["path"]) in SEG_MAP)
        pct   = 100 * n_hit / len(rows)
        flag  = "" if pct == 100 else " ⚠" if pct < 50 else " ✓"
        print(f'  {ds:<23} {split:<6} {n_hit:>7,} {len(rows):>7,} {pct:>8.1f}%{flag}')

def get_seg_mask(img_path):
    """
    Return a binary uint8 mask (1=animal, 0=background) derived from the
    pre-segmented image, or None if not in the cache.

    The segmented image has background pixels set to pure white (255,255,255).
    Mask = pixels that are NOT pure white.  Imperfect for white-furred/scaled
    animals, but good enough for SeaTurtles, Salamanders, and TexasLizards.
    """
    p = Path(img_path)
    try:
        rel_key = str(p.relative_to(_root))
    except ValueError:
        return None
    if rel_key not in SEG_MAP:
        return None
    seg = cv2.imread(str(SEG_MAP[rel_key]))
    if seg is None:
        return None
    # Background was painted to (255, 255, 255) — mark those as 0
    is_bg = (seg[:, :, 0] == 255) & (seg[:, :, 1] == 255) & (seg[:, :, 2] == 255)
    return (~is_bg).astype(np.uint8)

print('get_seg_mask() ready.')


## Section 2: Global Features (MiewID v3)

In [ ]:
# Cell 2.1: MiewID v3 Model

class MiewIDWrapper(nn.Module):
    """MiewID v3 wrapper for global feature extraction."""
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(
            "conservationxlabs/miewid-msv3",
            trust_remote_code=True
        )

    def forward(self, x):
        return self.backbone(x)


def get_global_model():
    """Initialize MiewID v3 model with multi-GPU support."""
    model = MiewIDWrapper()
    model.to(DEVICE)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    
    model.eval()
    return model

print("✓ MiewID v3 model wrapper defined")

In [ ]:
# Cell 2.2: Extract Global Features

def extract_global_features(model, dataset, image_size):
    """Extract L2-normalized global features with TTA (horizontal flip)."""
    dataset.transform = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    
    loader = DataLoader(
        dataset, 
        batch_size=BATCH_SIZE, 
        num_workers=NUM_WORKERS, 
        shuffle=False
    )
    
    all_features = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting global features", leave=False):
            images = batch[0].to(DEVICE)
            
            # TTA: original + horizontal flip
            with torch.cuda.amp.autocast():
                feats_sum = model(images) + model(torch.flip(images, dims=[3]))
            
            # L2 normalize
            feats_norm = torch.nn.functional.normalize(feats_sum.float(), p=2, dim=1)
            all_features.append(feats_norm.cpu().numpy())
    
    return np.concatenate(all_features)

print("✓ Global feature extraction function defined")

In [ ]:
# Cell 2.3: Cache Global Embeddings

global_features_cache = {}
model = get_global_model()

for species in test_meta["dataset"].unique():
    print(f"\n{'='*60}")
    print(f"Processing {species} - Global Features")
    
    cfg = SPECIES_CONFIG[species]
    cache_file = f"cache/embeddings/{species}_global.npy"
    
    # Check cache
    if os.path.exists(cache_file):
        print(f"  Loading cached embeddings: {cache_file}")
        features = np.load(cache_file)
    else:
        # Extract features
        sp_meta = test_meta[test_meta["dataset"] == species]
        sp_dataset = full_dataset.get_subset(sp_meta.index.values)
        
        features = extract_global_features(model, sp_dataset, cfg["image_size"])
        
        # Cache to disk
        np.save(cache_file, features)
        print(f"  Cached to {cache_file}")
    
    global_features_cache[species] = features
    print(f"  Shape: {features.shape}, Norm: {np.linalg.norm(features[0]):.3f}")

del model
torch.cuda.empty_cache()
print("\n✓ Global features extracted and cached")

In [ ]:
# Cell 2.3b: MegaDescriptor-L-384 Model
#
# Wildlife re-ID backbone trained on 37k+ individuals across 247 datasets.
# Distinct visual cues from MiewID → complementary representation.
# Loaded via timm from HuggingFace Hub (requires internet or pre-cached weights).

def get_mega_model():
    """Load MegaDescriptor-L-384 via timm (HuggingFace Hub)."""
    # num_classes=0 strips the classification head → returns 1536-dim feature embedding.
    # Without it, timm returns class logits (wrong dimension, wrong semantics).
    model = timm.create_model('hf-hub:BVRA/MegaDescriptor-L-384', pretrained=True, num_classes=0)
    model = model.eval().to(DEVICE)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    return model

print("✓ MegaDescriptor-L-384 model loader defined")


In [ ]:
# Cell 2.3c: Extract and Cache MegaDescriptor Features

# Swin-L at 384×384 does TWO forward passes per batch (TTA).  Use a smaller
# batch than the global BATCH_SIZE (64) to avoid OOM on P100 (16 GB).
# Increase to 64 if GPU memory allows; decrease to 16 if you see OOM errors.
MEGA_BATCH_SIZE = 32

def extract_mega_features(model, dataset, image_size=384):
    """Extract L2-normalized MegaDescriptor features with TTA (horizontal flip)."""
    dataset.transform = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])

    loader = DataLoader(
        dataset,
        batch_size=MEGA_BATCH_SIZE,
        num_workers=NUM_WORKERS,
        shuffle=False,
    )

    all_features = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting MegaDescriptor features", leave=False):
            images = batch[0].to(DEVICE)

            # TTA: original + horizontal flip
            with torch.cuda.amp.autocast():
                feats = model(images) + model(torch.flip(images, dims=[3]))

            feats_norm = torch.nn.functional.normalize(feats.float(), p=2, dim=1)
            all_features.append(feats_norm.cpu().numpy())

    return np.concatenate(all_features)


mega_features_cache = {}
mega_model = get_mega_model()

for species in test_meta["dataset"].unique():
    print(f"\n{'='*60}")
    print(f"Processing {species} - MegaDescriptor Features")

    cache_file = f"cache/embeddings/{species}_mega.npy"

    if os.path.exists(cache_file):
        print(f"  Loading cached embeddings: {cache_file}")
        features = np.load(cache_file)
    else:
        sp_meta = test_meta[test_meta["dataset"] == species]
        sp_dataset = full_dataset.get_subset(sp_meta.index.values)

        features = extract_mega_features(mega_model, sp_dataset, image_size=384)

        np.save(cache_file, features)
        print(f"  Cached to {cache_file}")

    mega_features_cache[species] = features
    print(f"  Shape: {features.shape}, Norm: {np.linalg.norm(features[0]):.3f}")

del mega_model
torch.cuda.empty_cache()
print("\n✓ MegaDescriptor features extracted and cached")


In [ ]:
# Cell 2.3d: Fuse MiewID + MegaDescriptor Embeddings
#
# Both are already L2-normalized per species.
# MiewID (2152-dim) ++ MegaDescriptor (1536-dim) → concat → re-normalize → 3688-dim.
# global_features_cache is updated in-place; downstream cells (QE, cosine sim,
# top-K selection) are unchanged — they only consume global_features_cache.

for species in list(global_features_cache.keys()):
    miew  = global_features_cache[species]          # (N, 2152) L2-normalized
    mega  = mega_features_cache[species]             # (N, 1536) L2-normalized
    fused = np.concatenate([miew, mega], axis=1)    # (N, 3688)
    fused = fused / (np.linalg.norm(fused, axis=1, keepdims=True) + 1e-8)
    global_features_cache[species] = fused
    print(f"  {species}: fused shape {fused.shape}, norm={np.linalg.norm(fused[0]):.3f}")

print("\n✓ Global features fused (MiewID + MegaDescriptor → 3688-dim)")


In [ ]:
# Cell 2.4: Query Expansion (Optional)

def query_expansion(features, k=5, alpha=0.5):
    """Query expansion via k-nearest neighbors averaging."""
    sim_matrix = np.dot(features, features.T)
    knn_indices = np.argsort(sim_matrix, axis=1)[:, -k:]
    
    expanded = np.zeros_like(features)
    for i in range(features.shape[0]):
        expanded[i] = features[i] + alpha * np.mean(features[knn_indices[i]], axis=0)
    
    return normalize(expanded, axis=1, norm="l2")


# Apply query expansion
global_features_expanded = {}
for species, features in global_features_cache.items():
    cfg = SPECIES_CONFIG[species]
    expanded = query_expansion(features, k=cfg["qe_k"])
    global_features_expanded[species] = expanded
    print(f"{species}: QE with k={cfg['qe_k']}")

print("✓ Query expansion applied to global features")

## Section 3: Local Features (SIFT, SuperPoint, ALIKED, DISK)

In [ ]:
# Cell 3.1: SIFT Extractor

class SIFTExtractor:
    """SIFT feature extractor using OpenCV (rotation-invariant)."""
    def __init__(self, max_keypoints=1000):
        self.max_keypoints = max_keypoints
        self.sift = cv2.SIFT_create(nfeatures=max_keypoints)
    
    def extract(self, image_path):
        """Extract SIFT keypoints and descriptors."""
        img = cv2.imread(str(image_path))
        if img is None:
            return None
        
        # Convert to grayscale for SIFT
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Detect and compute
        keypoints, descriptors = self.sift.detectAndCompute(gray, None)
        
        if descriptors is None or len(keypoints) < 4:
            return None
        
        # RootSIFT: L1-normalize then element-wise sqrt (HotSpotter's descriptor).
        # More discriminative for texture-heavy patterns (turtles, lizards).
        # SIFT descriptors are non-negative (gradient-magnitude histograms).
        descriptors /= (descriptors.sum(axis=1, keepdims=True) + 1e-7)
        descriptors = np.sqrt(descriptors)
        
        # Filter keypoints to animal region using segmentation mask.
        # SIFT runs on the ORIGINAL image (no white-bg boundary artifacts).
        # get_seg_mask() returns None for uncached images → all kpts kept.
        seg_mask = get_seg_mask(image_path)
        if seg_mask is not None:
            h, w = seg_mask.shape
            keep = [
                i for i, kp in enumerate(keypoints)
                if 0 <= int(kp.pt[1]) < h
                and 0 <= int(kp.pt[0]) < w
                and seg_mask[int(kp.pt[1]), int(kp.pt[0])] == 1
            ]
            if len(keep) >= 4:
                keypoints   = [keypoints[i] for i in keep]
                descriptors = descriptors[keep]
        
        # Convert keypoints to numpy array (x, y)
        kpts_array = np.array([kp.pt for kp in keypoints], dtype=np.float32)
        scores_array = np.array([kp.response for kp in keypoints], dtype=np.float32)
        
        return {
            'keypoints': kpts_array,
            'descriptors': descriptors.astype(np.float32),
            'scores': scores_array
        }

print("✓ SIFT extractor defined")

In [ ]:
# Cell 3.2: Feature Extractor Strategy

# SIMPLIFIED APPROACH: SIFT-Only Ensemble
# 
# Due to kornia compatibility issues (ALIKED/DISK not available in all versions),
# we use a proven, robust approach:
#
# 1. SIFT (OpenCV): Classical rotation-invariant features
#    - Works on all platforms
#    - 128-dim descriptors
#    - Matched with BFMatcher + Lowe's ratio test
#    - Excellent for rotation/scale invariance
#
# 2. Global (MiewID v3): Deep learning embeddings
#    - 2152-dim features
#    - Pre-trained on wildlife data
#    - Excellent for overall similarity
#
# Ensemble: Global (65-70%) + SIFT (30-35%)
# This provides a good balance of deep learning and classical CV approaches

print("✓ Using SIFT + MiewID v3 ensemble (robust, compatible approach)")

In [ ]:
# Cell 3.3: ALIKED Extractor (Kornia)

class ALIKEDExtractor:
    """ALIKED feature extractor via kornia (deformation-robust)."""
    def __init__(self, max_keypoints=1024, device='cuda'):
        self.device = device
        self.extractor = KF.ALIKED(max_num_keypoints=max_keypoints).to(device).eval()
    
    def extract(self, image_path):
        """Extract ALIKED keypoints and descriptors."""
        img = cv2.imread(str(image_path))
        if img is None:
            return None
        
        # Convert to grayscale tensor
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        tensor = torch.from_numpy(gray).float()[None, None] / 255.0
        tensor = tensor.to(self.device)
        
        with torch.no_grad():
            result = self.extractor(tensor)
        
        if result is None or len(result[0]['keypoints']) == 0:
            return None
        
        return {
            'keypoints': result[0]['keypoints'].cpu().numpy(),
            'descriptors': result[0]['descriptors'].cpu().numpy(),
            'scores': result[0]['scores'].cpu().numpy()
        }

print("✓ ALIKED extractor defined")

In [ ]:
# Cell 3.4: DISK Extractor (Kornia)

class DISKExtractor:
    """DISK feature extractor via kornia (dense keypoints)."""
    def __init__(self, max_keypoints=1024, device='cuda'):
        self.device = device
        self.max_keypoints = max_keypoints
        self.extractor = KF.DISK.from_pretrained('depth').to(device).eval()
    
    def extract(self, image_path):
        """Extract DISK keypoints and descriptors."""
        img = cv2.imread(str(image_path))
        if img is None:
            return None
        
        # Resize to fixed size to avoid padding issues
        # DISK works best with images divisible by 16
        target_size = 768  # 768 is divisible by 16
        img_resized = cv2.resize(img, (target_size, target_size))
        
        # Convert to RGB tensor
        img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
        tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).float()[None] / 255.0
        tensor = tensor.to(self.device)
        
        with torch.no_grad():
            try:
                # Call DISK detector
                result = self.extractor(tensor, n=self.max_keypoints)
                
                if 'keypoints' not in result or len(result['keypoints'][0]) == 0:
                    return None
                
                # Extract results
                keypoints = result['keypoints'][0].cpu().numpy()
                descriptors = result['descriptors'][0].cpu().numpy()
                
                # Get scores (detection confidence)
                if 'detection_scores' in result:
                    scores = result['detection_scores'][0].cpu().numpy()
                else:
                    scores = np.ones(len(keypoints), dtype=np.float32)
                
                if len(keypoints) < 4:
                    return None
                
                # Scale keypoints back to original image size
                h_orig, w_orig = img.shape[:2]
                keypoints[:, 0] *= w_orig / target_size
                keypoints[:, 1] *= h_orig / target_size
                
                return {
                    'keypoints': keypoints.astype(np.float32),
                    'descriptors': descriptors.astype(np.float32),
                    'scores': scores.astype(np.float32)
                }
            except Exception as e:
                # Silently return None on failure (some images may be incompatible)
                return None

print("✓ DISK extractor defined")

In [ ]:
# Cell 3.5: Extract Local Features per Species

def get_extractor(extractor_name, device='cuda'):
    """Factory function to create feature extractors."""
    if extractor_name == 'sift':
        return SIFTExtractor(max_keypoints=1000)
    else:
        raise ValueError(f"Unknown extractor: {extractor_name}. Only 'sift' is supported.")


def extract_local_features_for_dataset(species, extractor_name, dataset_meta, root_dir):
    """Extract local features for all images in a dataset split."""
    cache_file = f"cache/local_features/{species}_{extractor_name}.pkl"
    
    # Check cache
    if os.path.exists(cache_file):
        print(f"  Loading cached {extractor_name} features: {cache_file}")
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    
    # Extract features
    print(f"  Extracting {extractor_name} features for {len(dataset_meta)} images...")
    extractor = get_extractor(extractor_name, DEVICE)
    
    features_list = []
    for idx, row in tqdm(dataset_meta.iterrows(), total=len(dataset_meta), leave=False, desc=f"{extractor_name}"):
        img_path = os.path.join(root_dir, row['path'])
        feats = extractor.extract(img_path)
        features_list.append(feats)
    
    # Cache to disk
    with open(cache_file, 'wb') as f:
        pickle.dump(features_list, f)
    print(f"  Cached to {cache_file}")
    
    return features_list


# Extract local features for each species
local_features_cache = {}

for species in test_meta["dataset"].unique():
    print(f"\n{'='*60}")
    print(f"Processing {species} - Local Features (SIFT)")
    
    cfg = SPECIES_CONFIG[species]
    sp_meta = test_meta[test_meta["dataset"] == species]
    
    local_features_cache[species] = {}
    for extractor_name in cfg["local_extractors"]:
        features = extract_local_features_for_dataset(
            species, extractor_name, sp_meta, ROOT_DIR
        )
        local_features_cache[species][extractor_name] = features
        
        # Stats
        valid_count = sum(1 for f in features if f is not None)
        if valid_count > 0:
            avg_kpts = np.mean([len(f['keypoints']) for f in features if f is not None])
            print(f"    ✓ {extractor_name}: {valid_count}/{len(features)} valid, "
                  f"avg {avg_kpts:.0f} keypoints")
        else:
            print(f"    ⚠ {extractor_name}: No valid features extracted!")

print("\n✓ Local features extracted and cached")

In [ ]:
# Cell 3.6: Verify Local Features Cache

print("Local Features Summary:")
for species, extractors_dict in local_features_cache.items():
    print(f"\n{species}:")
    for extractor_name, features_list in extractors_dict.items():
        valid = sum(1 for f in features_list if f is not None)
        print(f"  {extractor_name}: {valid}/{len(features_list)} images with features")

print("\n✓ Local features verified")

## Section 4: Matching with LightGlue

In [ ]:
# Cell 4.1: SIFT Matcher (BFMatcher)

class SIFTMatcher:
    """SIFT matcher using BFMatcher with Lowe's ratio test."""
    def __init__(self, device='cuda'):
        # BFMatcher for SIFT (L2 norm)
        self.matcher = cv2.BFMatcher(cv2.NORM_L2)
    
    def match(self, feat0, feat1):
        """Match SIFT keypoints between two images."""
        if feat0 is None or feat1 is None:
            return {'matches': np.array([]), 'match_indices': np.array([]), 'confidence': np.array([])}
        
        if len(feat0['descriptors']) < 2 or len(feat1['descriptors']) < 2:
            return {'matches': np.array([]), 'match_indices': np.array([]), 'confidence': np.array([])}
        
        # kNN matching with k=2
        matches = self.matcher.knnMatch(feat0['descriptors'], feat1['descriptors'], k=2)
        
        # Lowe's ratio test (0.75 threshold)
        good_matches = []
        for pair in matches:
            if len(pair) == 2:
                m, n = pair
                if m.distance < 0.75 * n.distance:
                    good_matches.append(m)
        
        if len(good_matches) == 0:
            return {
                'matches': np.array([], dtype=int),
                'match_indices': np.array([], dtype=int),
                'confidence': np.array([], dtype=np.float32)
            }
        
        return {
            'matches': np.array([m.queryIdx for m in good_matches], dtype=int),
            'match_indices': np.array([m.trainIdx for m in good_matches], dtype=int),
            'confidence': 1.0 - np.array([m.distance for m in good_matches], dtype=np.float32) / 255.0
        }

print("✓ SIFT matcher defined (BFMatcher with Lowe's ratio test)")

In [ ]:
# Cell 4.2: LNBNN Matching (HotSpotter-style scoring)
#
# Replaces BFMatcher + Lowe's ratio test with Local Naive Bayes Nearest Neighbor.
#
# Key idea: for each query descriptor q, find k+1 nearest neighbors across the
# full candidate database.  The (k+1)th distance is the "background" reference.
# Image i accumulates: sum(bg_dist - match_dist) for each query kpt whose
# k nearest neighbors land in image i.  This naturally weights images that
# contain many close (distinctive) matches rather than just counting them.
#
# Still uses top-100 global cosine-similarity pre-selection to bound runtime.
#
# GPU optimisations:
#   - db and index tensors uploaded once per query (H2D); topk done on GPU
#   - Accumulation via scatter_add_ stays on GPU — no D2H of topk results
#   - Match-matrix update is vectorised numpy (no Python inner loop)
#   - argpartition (O(N)) replaces argsort (O(N log N)) for top-K selection
#   - astype(..., copy=False) skips copies when descriptors already float32

def lnbnn_match_scores(query_descs, db_descs_list, k=3):
    """
    LNBNN scoring: match one query image against a list of db images.

    Args:
        query_descs  : (Nq, D) float32 array (RootSIFT descriptors for query).
        db_descs_list: list of length n_db; each entry is an (Ni, D) float32
                       array or None (treated as missing → score stays 0).
        k            : number of nearest-neighbor votes per query descriptor.

    Returns:
        scores: (n_db,) float64 numpy array of non-negative LNBNN scores.
    """
    if query_descs is None or len(query_descs) < 4:
        return np.zeros(len(db_descs_list))

    # Collect valid (non-None, non-empty) db entries, preserve original indices
    valid = [
        (orig_i, d) for orig_i, d in enumerate(db_descs_list)
        if d is not None and len(d) >= 1
    ]
    if not valid:
        return np.zeros(len(db_descs_list))

    orig_indices, valid_descs = zip(*valid)

    # Flat database — copy=False avoids a redundant copy if already float32
    db_flat    = np.concatenate(valid_descs, axis=0).astype(np.float32, copy=False)
    # int64 required for scatter_add_ index tensor
    db_img_idx = np.concatenate(
        [np.full(len(d), oi, dtype=np.int64) for oi, d in valid], axis=0
    )  # (N_total,) maps each db descriptor row → original db_descs_list index

    q_np = query_descs.astype(np.float32, copy=False)

    # Upload db and index to GPU once (H2D); topk stays on GPU for scatter_add_
    db             = torch.from_numpy(db_flat).to(DEVICE)
    db_img_idx_gpu = torch.from_numpy(db_img_idx).to(DEVICE)   # int64 LongTensor
    kk             = min(k + 1, db.shape[0])
    n_votes        = kk - 1

    # Accumulation buffer lives on GPU — no D2H until the very end
    scores_gpu = torch.zeros(len(db_descs_list), dtype=torch.float64, device=DEVICE)

    # Chunked GPU kNN: caps peak memory to ~100 MB per chunk
    CHUNK = 256
    for s in range(0, len(q_np), CHUNK):
        q_chunk = torch.from_numpy(q_np[s:s + CHUNK]).to(DEVICE)
        d_chunk = torch.cdist(q_chunk, db, p=2)                  # (chunk, N_total)
        td, ti  = torch.topk(d_chunk, kk, dim=1, largest=False)
        del d_chunk, q_chunk

        bg = td[:, -1]                                            # (chunk,) background dist

        # GPU scatter_add_ — no need to copy topk results back to CPU
        for ki in range(n_votes):
            img_ids = db_img_idx_gpu[ti[:, ki]]                   # (chunk,) long
            deltas  = (bg - td[:, ki]).double()                   # (chunk,) float64
            pos     = deltas > 0
            scores_gpu.scatter_add_(0, img_ids[pos], deltas[pos])

        del td, ti

    del db, db_img_idx_gpu
    return scores_gpu.cpu().numpy()                               # single D2H: K floats


def compute_pairwise_matches_fast(features_list, extractor_type, species):
    """Pairwise LNBNN matching with top-K global candidate pre-selection."""
    n = len(features_list)
    cache_file = f"cache/match_scores/{species}_{extractor_type}_matches.npy"

    if os.path.exists(cache_file):
        print(f"  ✓ Loading cached {extractor_type} match scores: {cache_file}")
        return np.load(cache_file)

    print(f"  Computing {extractor_type} LNBNN matches for {n} images...")

    K = min(100, n)

    global_feats = global_features_expanded[species]
    global_sim   = np.dot(global_feats, global_feats.T)

    # argpartition is O(N) vs argsort O(N log N); top-K order doesn't matter here
    top_k_all = np.argpartition(global_sim, -K, axis=1)[:, -K:]  # (N, K)

    match_matrix = np.zeros((n, n), dtype=np.float32)
    np.fill_diagonal(match_matrix, 1.0)

    for i in tqdm(range(n), desc=f"LNBNN {extractor_type}"):
        if features_list[i] is None:
            continue

        candidates = top_k_all[i]

        # Pass None for j==i so LNBNN never matches descriptors against themselves
        db_descs_list = [
            features_list[j]['descriptors'] if (features_list[j] is not None and j != i) else None
            for j in candidates
        ]

        raw_scores = lnbnn_match_scores(
            features_list[i]['descriptors'],
            db_descs_list,
        )

        # Vectorised match-matrix update — replaces the K-iteration Python loop
        valid_mask = candidates != i
        js         = candidates[valid_mask]
        scores_01  = (1.0 - np.exp(-raw_scores[valid_mask] / 20.0)).astype(np.float32)

        # np.maximum with assignment: 2 numpy calls instead of 100 Python iters
        match_matrix[i, js] = np.maximum(match_matrix[i, js], scores_01)
        match_matrix[js, i] = np.maximum(match_matrix[js, i], scores_01)

    np.save(cache_file, match_matrix)
    print(f"  ✓ Cached to {cache_file}")

    return match_matrix

print("✓ LNBNN matching defined (HotSpotter-style scoring)")


In [ ]:
# Cell 4.3: RANSAC Verification and Match Scoring

def ransac_filter(kpts0, kpts1, matches, threshold=3.0):
    """RANSAC geometric verification of matches."""
    if len(matches['matches']) < 4:
        return np.array([], dtype=int)
    
    pts0 = kpts0[matches['matches']]
    pts1 = kpts1[matches['match_indices']]
    
    # Estimate fundamental matrix with RANSAC
    _, mask = cv2.findFundamentalMat(
        pts0, pts1, cv2.FM_RANSAC, threshold, confidence=0.99
    )
    
    if mask is None:
        return np.array([], dtype=int)
    
    return np.where(mask.ravel())[0]


def score_local_match(matches, feat0, feat1):
    """Convert matches to similarity score [0, 1]."""
    if feat0 is None or feat1 is None:
        return 0.0
    
    if len(matches['matches']) == 0:
        return 0.0
    
    # RANSAC geometric verification
    inlier_indices = ransac_filter(
        feat0['keypoints'], 
        feat1['keypoints'], 
        matches
    )
    
    num_inliers = len(inlier_indices)
    if num_inliers == 0:
        return 0.0
    
    # Average confidence of inlier matches
    avg_confidence = np.mean(matches['confidence'][inlier_indices])
    
    # Sigmoid-like normalization (plateaus at ~50 matches)
    match_score = 1.0 - np.exp(-num_inliers / 20.0)
    
    # Combine match count and confidence
    final_score = match_score * avg_confidence
    
    return float(final_score)

print("✓ RANSAC verification and match scoring functions defined")

In [ ]:
# Cell 4.4: Compute and Cache Match Scores

match_scores_cache = {}

for species in test_meta["dataset"].unique():
    print(f"\n{'='*60}")
    print(f"Processing {species} - Match Scores")
    
    cfg = SPECIES_CONFIG[species]
    match_scores_cache[species] = {}
    
    for extractor_name in cfg["local_extractors"]:
        features_list = local_features_cache[species][extractor_name]
        
        # Use fast top-K matching strategy
        match_matrix = compute_pairwise_matches_fast(features_list, extractor_name, species)
        match_scores_cache[species][extractor_name] = match_matrix
        
        # Stats
        non_diag = match_matrix[~np.eye(match_matrix.shape[0], dtype=bool)]
        print(f"    ✓ {extractor_name}: Shape {match_matrix.shape}, "
              f"Mean score: {non_diag.mean():.3f}, "
              f"Non-zero pairs: {(non_diag > 0).sum():,}")

print("\n✓ Match scores computed and cached")

## Section 5: Ensemble Voting

In [ ]:
# Cell 5.1: Ensemble Scoring Function

def compute_ensemble_similarity_matrix(species):
    """Compute weighted ensemble similarity matrix."""
    cfg = SPECIES_CONFIG[species]
    
    # Global similarity (cosine)
    global_feats = global_features_expanded[species]
    global_sim = np.dot(global_feats, global_feats.T)
    
    # Start with weighted global similarity
    ensemble_sim = cfg["global_weight"] * global_sim
    
    # Add weighted local match scores
    for extractor_name, weight in cfg["local_weights"].items():
        local_sim = match_scores_cache[species][extractor_name]
        ensemble_sim += weight * local_sim
    
    return ensemble_sim

print("✓ Ensemble scoring function defined")

In [ ]:
# Cell 5.2: Compute Ensemble Scores for All Species

ensemble_similarity_cache = {}

for species in test_meta["dataset"].unique():
    print(f"\nComputing ensemble scores for {species}...")
    ensemble_sim = compute_ensemble_similarity_matrix(species)
    ensemble_similarity_cache[species] = ensemble_sim
    
    # Stats
    print(f"  Shape: {ensemble_sim.shape}")
    print(f"  Mean similarity: {ensemble_sim.mean():.3f}")
    print(f"  Std similarity: {ensemble_sim.std():.3f}")
    print(f"  Min/Max: {ensemble_sim.min():.3f} / {ensemble_sim.max():.3f}")

print("\n✓ Ensemble similarity matrices computed")

In [ ]:
# Cell 5.3: Weighted Voting Summary

print("Ensemble Weights Summary:\n")
for species, cfg in SPECIES_CONFIG.items():
    print(f"{species}:")
    print(f"  Global (MiewID v3): {cfg['global_weight']:.0%}")
    for extractor, weight in cfg['local_weights'].items():
        print(f"  {extractor.upper()}: {weight:.0%}")
    print()

print("✓ Ensemble voting configured")

## Section 6: Clustering and Submission

In [ ]:
# Cell 6.1: 2-Phase Clustering (Known + Unknown)

def cluster_with_ensemble_scores(species, similarity_matrix, image_ids, threshold):
    """Cluster images using ensemble similarity scores."""
    print(f"\n  Clustering {species}:")
    print(f"    Threshold: {threshold}")
    print(f"    Images: {len(image_ids)}")
    
    # Convert similarity to distance
    dist_matrix = np.clip(1.0 - similarity_matrix, 0, 1)
    
    # Agglomerative clustering
    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=threshold,
    )
    labels = clustering.fit_predict(dist_matrix)
    
    n_clusters = len(np.unique(labels))
    print(f"    ✅ Found {n_clusters} individuals")
    
    # Format cluster labels
    cluster_labels = [f"cluster_{species}_{lbl}" for lbl in labels]
    
    return pd.DataFrame({
        "image_id": image_ids,
        "cluster": cluster_labels
    })

print("✓ Clustering function defined")

In [ ]:
# Cell 6.2: Generate Clusters for All Species

results = []

print("="*60)
print("Clustering with Ensemble Scores")
print("="*60)

for species in test_meta["dataset"].unique():
    cfg = SPECIES_CONFIG[species]
    sp_meta = test_meta[test_meta["dataset"] == species]
    
    # Get ensemble similarity matrix
    similarity_matrix = ensemble_similarity_cache[species]
    
    # Cluster
    cluster_df = cluster_with_ensemble_scores(
        species,
        similarity_matrix,
        sp_meta["image_id"].values,
        cfg["threshold_cluster"]
    )
    
    results.append(cluster_df)

print("\n✓ All species clustered")

In [ ]:
# Cell 6.3: Generate Submission CSV

# Combine all results
predictions = pd.concat(results, ignore_index=True)

# Load sample submission
sample_sub = pd.read_csv(os.path.join(ROOT_DIR, "sample_submission.csv"))

# Map predictions to sample submission format
pred_map = dict(zip(predictions["image_id"], predictions["cluster"]))
sample_sub["cluster"] = sample_sub["image_id"].map(pred_map).fillna("cluster_error_0")

# Save submission
sample_sub.to_csv("submission.csv", index=False)

print("\n" + "="*60)
print("🏆 Submission Generated!")
print("="*60)
print(f"Total predictions: {len(sample_sub):,}")
print(f"Total clusters: {sample_sub['cluster'].nunique():,}")
print(f"\nBreakdown by species:")
for species in test_meta["dataset"].unique():
    sp_clusters = predictions[predictions["cluster"].str.contains(species)]["cluster"].nunique()
    print(f"  {species}: {sp_clusters} clusters")

print("\n✓ submission.csv saved")
sample_sub.head(10)